<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

Para modelos baseados em árvores, como Random Forest e AdaBoost, a normalização dos dados não é estritamente necessária. Esses algoritmos não são sensíveis à escala das variáveis, pois as divisões são feitas com base em thresholds dos próprios valores das features. Portanto, normalizar ou padronizar os dados não traz benefícios significativos para esses modelos.

In [1]:
# TODO: implemente load_data
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

fashion = fetch_openml('Fashion-MNIST', as_frame=False)
x = fashion.data 
y = fashion.target.astype(int)

x_train,x_test,y_train,y_test = train_test_split(x ,y , stratify = y, random_state= 42, test_size= 0.2)





# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

A função evaluate utiliza a acurácia como métrica principal, que representa a proporção de previsões corretas em relação ao total de exemplos. É uma métrica adequada para problemas balanceados, mas pode ser complementada por outras métricas (como precisão, recall e F1-score) em casos de classes desbalanceadas.

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier

def train_random_forest(X_train, y_train, seed):
    model = RandomForestClassifier(n_estimators= 200, max_depth = 20, max_leaf_nodes= 16, min_samples_leaf = 5, max_features= 'sqrt', random_state=seed)
    model.fit(X_train, y_train)
    return model
def train_adaboost(X_train, y_train, seed):
    model = AdaBoostClassifier(random_state=seed)
    model.fit(X_train, y_train)
    return model


# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

A função evaluate utiliza a acurácia como métrica principal, que representa a proporção de previsões corretas em relação ao total de exemplos. É uma métrica adequada para problemas balanceados, mas pode ser complementada por outras métricas (como precisão, recall e F1-score) em casos de classes desbalanceadas.

In [3]:
def evaluate(model, X_test, y_test):
    from sklearn.metrics import accuracy_score
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return acc

**Adicione seu texto de solução aqui**.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

Em qual profundidade começa o overfitting?
O overfitting geralmente começa a ocorrer quando a profundidade da árvore é muito alta, permitindo que o modelo memorize os dados de treino. No caso do Random Forest, profundidades acima de 20 podem começar a apresentar sinais de overfitting, mas isso depende do dataset. É importante monitorar a diferença entre a acurácia de treino e teste para identificar esse ponto.

Quando max_depth=None, a árvore cresce até que todas as folhas sejam puras (ou contenham menos que o número mínimo de amostras para divisão). Isso faz com que o modelo memorize completamente os dados de treino, resultando em 100% de acurácia nesse conjunto, mas geralmente com pior desempenho em dados não vistos (overfitting).

In [4]:
def run_pipeline(model_type="rf", seed=42):
   
    if model_type == "rf":
        model = train_random_forest(x_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(x_train, y_train, seed)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'")
    
   
    acc = evaluate(model,x_test,y_test)
    return acc
    

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

Após rodar os experimentos, o modelo Random Forest geralmente apresenta melhor desempenho inicial em termos de acurácia, precisão, recall e F1-score, quando comparado ao AdaBoost. Isso ocorre porque o Random Forest é mais robusto e menos sensível a ruídos, enquanto o AdaBoost pode ser mais afetado por outliers.

In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
def run_and_report(seed=42):
    
    acc_rf = run_pipeline(model_type="rf", seed=seed)
    model_rf = train_random_forest(x_train, y_train, seed)
    y_pred_rf = model_rf.predict(x_test)
    prec_rf = precision_score(y_test, y_pred_rf, average='weighted')
    rec_rf = recall_score(y_test, y_pred_rf, average='weighted')
    f1_rf = f1_score(y_test, y_pred_rf, average='weighted')
    
    
    acc_ab = run_pipeline(model_type="ab", seed=seed)
    model_ab = train_adaboost(x_train, y_train, seed)
    y_pred_ab = model_ab.predict(x_test)
    prec_ab = precision_score(y_test, y_pred_ab, average='weighted')
    rec_ab = recall_score(y_test, y_pred_ab, average='weighted')
    f1_ab = f1_score(y_test, y_pred_ab, average='weighted')
    
    print("Random Forest:")
    print(f"Acurácia: {acc_rf:.4f} | Precisão: {prec_rf:.4f} | Recall: {rec_rf:.4f} | F1-Score: {f1_rf:.4f}")
    print("AdaBoost:")
    print(f"Acurácia: {acc_ab:.4f} | Precisão: {prec_ab:.4f} | Recall: {rec_ab:.4f} | F1-Score: {f1_ab:.4f}")
    
    if acc_rf > acc_ab:
        print("\nRandom Forest apresentou melhor desempenho inicial.")
    elif acc_ab > acc_rf:
        print("\nAdaBoost apresentou melhor desempenho inicial.")
    else:
        print("\nOs modelos tiveram desempenho inicial igual.")
run_and_report(seed=42)

Random Forest:
Acurácia: 0.7559 | Precisão: 0.7776 | Recall: 0.7559 | F1-Score: 0.7255
AdaBoost:
Acurácia: 0.4909 | Precisão: 0.4717 | Recall: 0.4909 | F1-Score: 0.4448

Random Forest apresentou melhor desempenho inicial.


# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

- Ao executar o pipeline com diferentes seeds (por exemplo, 42 e 7), observe se as métricas (acurácia, precisão, recall, F1-score) dos modelos mudam.
- Se os resultados forem idênticos ou muito próximos, o experimento é considerado reprodutível, pois o uso do parâmetro `random_state` garante que a divisão dos dados e a inicialização dos modelos sejam controladas.
- Pequenas variações podem ocorrer devido à natureza estocástica de alguns algoritmos, mas grandes diferenças indicariam falta de controle de aleatoriedade.

**Exemplo de resposta:**

- Os resultados mudaram? As métricas apresentaram pequenas variações entre as seeds, mas os resultados foram semelhantes.
- O experimento é reprodutível? Sim, pois ao definir o parâmetro `random_state` em todas as etapas (divisão dos dados e modelos), garantimos que o experimento possa ser repetido e os resultados serão consistentes, salvo pequenas flutuações inerentes ao processo de aprendizado de máquina.

Ao executar o pipeline com diferentes seeds (por exemplo, 42 e 7), as métricas dos modelos podem apresentar pequenas variações, mas tendem a ser muito próximas, indicando estabilidade dos resultados.

Sim, o experimento é reprodutível, pois o uso do parâmetro random_state tanto na divisão dos dados quanto na inicialização dos modelos garante que os resultados possam ser replicados. Pequenas diferenças podem ocorrer devido à aleatoriedade residual, mas não são significativas.

In [ ]:
# Execute o pipeline com diferentes seeds e compare os resultados
print('Resultados com seed=42:')
run_and_report(seed=42)
print('\nResultados com seed=7:')
run_and_report(seed=7)

Resultados com seed=42:


NameError: name 'teste_dos_modelos' is not defined

**Solução**:

- Ao comparar a acurácia em treino e teste para o Random Forest, normalmente a acurácia no treino é maior. Se a diferença for significativa, isso indica overfitting.
- O Random Forest tende a sofrer mais com overfitting do que o AdaBoost, especialmente se não houver limitação de profundidade ou número de árvores.
- No entanto, ambos os modelos podem ser ajustados para reduzir o overfitting por meio de hiperparâmetros como `max_depth`, `n_estimators` e `min_samples_leaf`.

In [ ]:

def compare_train_test_accuracy(seed=42):
    from sklearn.metrics import accuracy_score
    
    from sklearn.datasets import fetch_openml
    from sklearn.model_selection import train_test_split
    fashion = fetch_openml('Fashion-MNIST', as_frame=False)
    x = fashion.data 
    y = fashion.target.astype(int)
    x_train, x_test, y_train, y_test = train_test_split(x, y, stratify=y, random_state=seed, test_size=0.2)
  
    model = train_random_forest(x_train, y_train, seed)
    
    y_pred_train = model.predict(x_train)
    acc_train = accuracy_score(y_train, y_pred_train)
   
    y_pred_test = model.predict(x_test)
    acc_test = accuracy_score(y_test, y_pred_test)
    print(f'Acurácia treino: {acc_train:.4f}')
    print(f'Acurácia teste: {acc_test:.4f}')
    if acc_train - acc_test > 0.05:
        print('Há indícios de overfitting.')
    else:
        print('Não há indícios claros de overfitting.')
compare_train_test_accuracy(seed=42)

# Questão 8: Variação de hiperparâmetro n_estimators em ambos os modelos
def test_n_estimators_variation(seed=42):
    from sklearn.metrics import accuracy_score
    from sklearn.datasets import fetch_openml
    from sklearn.model_selection import train_test_split
    fashion = fetch_openml('Fashion-MNIST', as_frame=False)
    x = fashion.data 
    y = fashion.target.astype(int)
    x_train, x_test, y_train, y_test = train_test_split(x, y, stratify=y, random_state=seed, test_size=0.2)
    
    print('Random Forest:')
    for n in [10, 50, 100, 200]:
        model = RandomForestClassifier(n_estimators=n, max_depth=20, max_leaf_nodes=16, min_samples_leaf=5, max_features='sqrt', random_state=seed)
        model.fit(x_train, y_train)
        acc = accuracy_score(y_test, model.predict(x_test))
        print(f'n_estimators={n}: Acurácia={acc:.4f}')
    
    print('\nAdaBoost:')
    for n in [10, 50, 100, 200]:
        model = AdaBoostClassifier(n_estimators=n, random_state=seed)
        model.fit(x_train, y_train)
        acc = accuracy_score(y_test, model.predict(x_test))
        print(f'n_estimators={n}: Acurácia={acc:.4f}')
test_n_estimators_variation(seed=42)

In [ ]:
#- Ao variar o hiperparâmetro `n_estimators`, o desempenho do Random Forest tende a estabilizar após certo ponto, enquanto o AdaBoost pode ser mais sensível a esse parâmetro.
#- O AdaBoost geralmente apresenta maior variação de desempenho conforme o número de estimadores aumenta, pois é mais suscetível a overfitting e a ruídos nos dados.
#- O Random Forest é mais robusto a mudanças em `n_estimators`, mas pode apresentar ganhos marginais até certo limite.

**Solução**:

1. **A acurácia é suficiente para avaliar os modelos?**
   - Não. A acurácia pode ser insuficiente em problemas com classes desbalanceadas. É importante analisar também precisão, recall e F1-score para uma avaliação mais completa.

2. **Como você garante que o resultado não ocorreu por acaso?**
   - Utilizando validação cruzada, múltiplas execuções com diferentes seeds e controlando a aleatoriedade com `random_state`. Isso reduz o risco de resultados ocasionais.

3. **Cite dois possíveis problemas metodológicos neste experimento.**
   - Não realizar validação cruzada, confiando apenas em uma divisão treino/teste.
   - Não analisar o impacto de hiperparâmetros ou não controlar a aleatoriedade.

4. **O pipeline implementado é confiável? Justifique.**
   - Sim, pois utiliza controle de aleatoriedade, separação estratificada dos dados e funções bem definidas para cada etapa. Poderia ser aprimorado com validação cruzada e análise de variância dos resultados.

In [ ]:
# TODO: implemente load_data